# 10 · So sánh Validation và khóa lựa chọn Tuned V2

Mặc định chỉ cập nhật báo cáo. Project Leader nghiệm thu manifest trước khi mở Test.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, subprocess, uuid
DRIVE_FOLDER_ID = os.environ["PHISHSHIELD_DRIVE_FOLDER_ID"]
DRIVE_ROOT = None
RUNTIME = Path("/content/phishshield-ai-run")
# Nếu dùng shortcut, chọn đường dẫn nhìn thấy trong bảng Files của Colab.
candidate = "/content/drive/MyDrive/phishshield-ai"  # @param {type:"string"}
if not candidate or not Path(candidate).resolve().is_relative_to(Path("/content/drive")):
    raise ValueError("Chọn folder đã mount. Có thể thêm shortcut vào My Drive trước.")


### Xác minh Folder ID

Notebook đối chiếu metadata của mount. Nếu mount thiếu metadata, Drive API tạo một tệp thử trong folder có đúng ID và đối chiếu nội dung qua filesystem. Nếu xác minh thất bại, kiểm tra quyền Editor và shortcut rồi chạy lại.

In [ ]:
def resolve_drive_root(candidate=None, service=None):
    candidate = candidate or os.environ.get("PHISHSHIELD_DRIVE_ROOT")
    if not candidate:
        raise ValueError("Chọn đường dẫn mount thực tế hoặc shortcut trong My Drive.")
    root = Path(candidate).resolve(strict=True)
    if not root.is_dir():
        raise ValueError("Cần chọn thư mục Drive.")
    try:
        actual = (Path(str(root) + ":user.drive.id").read_text() if os.name == "nt"
                  else os.getxattr(root, "user.drive.id").decode())
    except (AttributeError, OSError):
        actual = None
    if actual == DRIVE_FOLDER_ID:
        return root
    if not service:
        raise ValueError("Chưa xác minh được Folder ID. Cấp quyền Drive API trong notebook.")
    # Prove the mount points to the API parent by reading a unique API-created file.
    from googleapiclient.http import MediaInMemoryUpload
    import time
    files = service.files()
    meta = files.get(fileId=DRIVE_FOLDER_ID, fields="mimeType,capabilities(canAddChildren)",
                     supportsAllDrives=True).execute()
    if meta["mimeType"] != "application/vnd.google-apps.folder" or not meta["capabilities"]["canAddChildren"]:
        raise PermissionError("Cần quyền ghi trong folder chính thức.")
    key = ".phishshield-probe-" + uuid.uuid4().hex
    value = os.urandom(32)
    probe = files.create(body={"name": key, "parents": [DRIVE_FOLDER_ID]},
                         media_body=MediaInMemoryUpload(value), supportsAllDrives=True, fields="id").execute()
    try:
        for _ in range(30):
            if (root / key).is_file() and (root / key).read_bytes() == value:
                return root
            time.sleep(1)
        raise ValueError("Sai đường dẫn hoặc mount chưa đồng bộ. Kiểm tra shortcut trong My Drive.")
    finally:
        files.delete(fileId=probe["id"], supportsAllDrives=True).execute()

try:
    DRIVE_ROOT = resolve_drive_root(candidate)
except ValueError:
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    creds, _ = google.auth.default()
    DRIVE_ROOT = resolve_drive_root(candidate, build("drive", "v3", credentials=creds))
print("Workspace đã xác minh:", DRIVE_ROOT)

In [ ]:
# Chạy một bản sao mã nguồn trong runtime. Mọi bản ghi bền vững đi vào folder đã xác minh.
import importlib.util
spec = importlib.util.spec_from_file_location("verified_storage", DRIVE_ROOT / "src/phishshield/utils/storage.py")
storage = importlib.util.module_from_spec(spec)
spec.loader.exec_module(storage)
storage.validate_drive_access(DRIVE_ROOT)
storage.sync_source_to_runtime(DRIVE_ROOT, RUNTIME)
os.environ.update(PHISHSHIELD_DRIVE_ROOT=str(DRIVE_ROOT), PHISHSHIELD_RUNTIME_ROOT=str(RUNTIME))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv==0.11.11"], check=True)
extras = ["trees"]
args = [s for x in extras for s in ("--extra", x)]
subprocess.run([sys.executable, "-m", "uv", "sync", "--project", str(RUNTIME), "--python", "3.11", "--frozen", *args], check=True)
python = str(RUNTIME / ".venv/bin/python")

In [ ]:
START_MODEL_SELECTION = True  # @param {type:"boolean"}
START_TEST_EVALUATION = True  # @param {type:"boolean"}
command = [python, str(RUNTIME / "scripts/workflow.py"), "10"]

def run_action(action):
    print(f"Đang chạy bước {action}", flush=True)
    result = subprocess.run(
        [*command, "--action", action], capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.returncode:
        raise RuntimeError(f"Bước {action} dừng với mã {result.returncode}\n{result.stderr.rstrip()}")
    if result.stderr:
        print(result.stderr.rstrip())

run_action("report")
if START_MODEL_SELECTION:
    run_action("select")
if START_TEST_EVALUATION:
    run_action("test")